In [ ]:
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

CONTROL_TABLE = "ctl_monthly_load"
START_PERIOD = date(2019, 1, 1)
END_PERIOD = date(2026, 6, 1)
BASE_URL = "https://transtats.bts.gov/PREZIP/"
FILE_PREFIX = "On_Time_Reporting_Carrier_On_Time_Performance_1987_present_"

schema = StructType([
    StructField("year", IntegerType(), False),
    StructField("month", IntegerType(), False),
    StructField("period_key", StringType(), False),
    StructField("file_name", StringType(), False),
    StructField("source_url", StringType(), False),
    StructField("download_status", StringType(), False),
    StructField("load_status", StringType(), False),
    StructField("last_updated", TimestampType(), True),
])

if spark.catalog.tableExists(CONTROL_TABLE):
    print(f"{CONTROL_TABLE} already exists. No changes made.")
else:
    rows = []
    year, month = START_PERIOD.year, START_PERIOD.month

    while (year, month) <= (END_PERIOD.year, END_PERIOD.month):
        file_name = f"{FILE_PREFIX}{year}_{month}.zip"
        rows.append((
            year,
            month,
            f"{year}-{month:02d}",
            file_name,
            f"{BASE_URL}{file_name}",
            "pending",
            "pending",
            None,
        ))
        month += 1
        if month > 12:
            year, month = year + 1, 1

    (
        spark.createDataFrame(rows, schema)
        .withColumn("last_updated", F.current_timestamp())
        .write
        .format("delta")
        .saveAsTable(CONTROL_TABLE)
    )
    print(f"Created {CONTROL_TABLE}.")

control = spark.table(CONTROL_TABLE)
summary = control.agg(
    F.count("*").alias("months"),
    F.min("period_key").alias("first_period"),
    F.max("period_key").alias("last_period"),
).first()

print(f"Months: {summary['months']}")
print(f"First period: {summary['first_period']}")
print(f"Last period: {summary['last_period']}")

Created ctl_monthly_load.
Months: 90
First period: 2019-01
Last period: 2026-06


In [ ]:
from pyspark.sql import functions as F

LOOKUP_PATH = "Files/raw/bts_lookups"

lookups = {
    "L_AIRLINE_ID.csv": "bronze_lkp_airline_id",
    "L_AIRPORT_ID.csv": "bronze_lkp_airport_id",
}

available = {f.name for f in notebookutils.fs.ls(LOOKUP_PATH)}
missing_files = [f for f in lookups if f not in available]
if missing_files:
    raise FileNotFoundError(f"Missing in {LOOKUP_PATH}: {missing_files}. Found: {sorted(available)}")

for file_name, table_name in lookups.items():
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("quote", '"')
        .option("escape", '"')
        .csv(f"{LOOKUP_PATH}/{file_name}")
    )

    df = df.toDF(*[c.strip().lower() for c in df.columns])

    df = (
        df
        .withColumn("_source_file", F.lit(file_name))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)

    saved = spark.table(table_name)
    print(f"{table_name}: {saved.count():,} rows | columns: {[c for c in saved.columns if not c.startswith('_')]}")
    for r in saved.limit(3).collect():
        print(f"  {r['code']} -> {r['description']}")

In [ ]:
from pyspark.sql import Row

spark.sql("DROP TABLE IF EXISTS lh_gold.dbo.measures")

spark.createDataFrame([Row(measure_home=1)]) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("lh_gold.dbo.measure_table")

print(spark.table("lh_gold.dbo.measure_table").count(), "row created in lh_gold.dbo.measure_table")

1 row created in lh_gold.dbo.measure_table
